# Notebook 6: Econometric Modelling

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Primary Modelling Period:** April 2017 – December 2025

**Unit of Analysis:** Food subclass × month

## Notebook Overview

This notebook develops the econometric component of the research using the
balanced food-subclass panel prepared in Notebook 5.

The econometric analysis has two purposes:

1. estimate a symmetric exchange-rate pass-through benchmark using ARDL models;
2. estimate asymmetric pass-through using nonlinear ARDL models that separate Rand depreciation and appreciation movements.

Models are estimated separately for each eligible food subclass. This allows the magnitude, direction and timing of exchange-rate pass-through to diffe across food categories.

The notebook covers model eligibility testing, stationarity analysis, lag selection and model estimation. Detailed diagnostics and research
interpretation will be completed in Notebook 7.

In [10]:
# import required libraries
from pathlib import Path

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.ardl import ARDL, UECM, ardl_select_order
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)


In [11]:
# load the econometric dataset
data_path = Path("../data/processed/econometric_model_data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Econometric dataset not found: {data_path.resolve()}"
    )

econometric_data = pd.read_csv(
    data_path,
    parse_dates=["Date"]
)

econometric_data = (
    econometric_data
    .sort_values(["SubclassDescription", "Date"])
    .reset_index(drop=True)
)

print(f"Dataset shape: {econometric_data.shape}")
print(
    "Period:",
    econometric_data["Date"].min().date(),
    "to",
    econometric_data["Date"].max().date()
)
print(
    "Food subclasses:",
    econometric_data["SubclassDescription"].nunique()
)

Dataset shape: (4830, 15)
Period: 2017-04-01 to 2025-12-01
Food subclasses: 46


In [12]:
# validate the modelling structure
required_columns = [
    "Date",
    "GroupDescription",
    "ClassDescription",
    "SubclassDescription",
    "Subclass_Weight",
    "CPI",
    "Log_CPI",
    "Food_Inflation_Pct",
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

missing_columns = sorted(
    set(required_columns) - set(econometric_data.columns)
)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

duplicate_count = econometric_data.duplicated(
    subset=["Date", "SubclassDescription"]
).sum()

missing_value_count = (
    econometric_data[required_columns]
    .isna()
    .sum()
    .sum()
)

subclass_month_counts = (
    econometric_data
    .groupby("SubclassDescription")["Date"]
    .nunique()
)

exchange_rate_columns = [
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

maximum_monthly_fx_values = (
    econometric_data
    .groupby("Date")[exchange_rate_columns]
    .nunique()
    .max()
    .max()
)

if duplicate_count != 0:
    raise ValueError("Duplicate subclass-month observations detected.")

if missing_value_count != 0:
    raise ValueError("Missing values detected in required variables.")

if subclass_month_counts.nunique() != 1:
    raise ValueError("Food subclasses do not have equal time coverage.")

if maximum_monthly_fx_values != 1:
    raise ValueError("Inconsistent exchange-rate values detected within months.")

validation_summary = pd.Series({
    "Observations": len(econometric_data),
    "Food subclasses": econometric_data[
        "SubclassDescription"
    ].nunique(),
    "Unique months": econometric_data["Date"].nunique(),
    "Minimum months per subclass": subclass_month_counts.min(),
    "Maximum months per subclass": subclass_month_counts.max(),
    "Duplicate subclass-month rows": duplicate_count,
    "Missing required values": missing_value_count,
    "Maximum FX values within a month": maximum_monthly_fx_values
})

validation_summary.to_frame(name="Value")

,Value
Observations,4830
Food subclasses,46
Unique months,105
Minimum months per subclass,105
Maximum months per subclass,105
Duplicate subclass-month rows,0
Missing required values,0
Maximum FX values within a month,1


### Modelling-Structure Validation

The validation confirms that the econometric dataset contains 4,830
observations representing 46 food subclasses across 105 common months.

Every subclass has complete coverage from April 2017 to December 2025. No duplicate subclass-month observations or missing modelling values were identified.

The exchange-rate variables are also consistent within each month, confirming that all food subclasses are matched to the same monthly macroeconomic series.

The dataset therefore satisfies the structural requirements for
subclass-specific time-series estimation.

### Econometric Modelling Framework

Two related econometric specifications will be estimated for each food
subclass.

### Symmetric ARDL Benchmark

The symmetric ARDL model uses log food CPI as the dependent variable and the log USD/ZAR exchange rate as the principal explanatory variable.

This specification assumes that Rand depreciation and appreciation have equal but opposite effects on food prices. It provides the conventional benchmark against which the asymmetric model can be assessed.

### Asymmetric NARDL Model

The NARDL specification replaces the single exchange-rate variable with its cumulative positive and negative components:

- `ExchangeRate_Positive_Cumulative_Pct` represents cumulative Rand
  depreciation.
- `ExchangeRate_Negative_Cumulative_Pct` represents cumulative Rand
  appreciation and remains negatively signed.

This decomposition allows depreciation and appreciation to have different short-run and long-run relationships with food prices.

### Dependent Variable

`Log_CPI` is used as the level-form dependent variable in the ARDL and NARDL specifications. Its first difference represents monthly food-price inflation.

Before estimating either model, the integration order of each variable must be assessed. ARDL bounds-testing methods permit a mixture of I(0) and I(1) variables but are not valid when any model variable is integrated of order two, I(2).

In [13]:
# stationarity-testing function
def run_stationarity_tests(
    series,
    variable_name,
    significance_level=0.05
):
    clean_series = (
        pd.Series(series)
        .dropna()
        .astype(float)
    )

    transformations = {
        "Level": clean_series,
        "First difference": clean_series.diff().dropna()
    }

    test_results = []

    for transformation, transformed_series in transformations.items():
        adf_result = adfuller(
            transformed_series,
            regression="c",
            autolag="AIC"
        )

        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=InterpolationWarning
            )

            kpss_result = kpss(
                transformed_series,
                regression="c",
                nlags="auto"
            )

        test_results.append({
            "Variable": variable_name,
            "Transformation": transformation,
            "Observations": len(transformed_series),
            "ADF_Statistic": adf_result[0],
            "ADF_P_Value": adf_result[1],
            "ADF_Lags": adf_result[2],
            "ADF_Stationary": (
                adf_result[1] < significance_level
            ),
            "KPSS_Statistic": kpss_result[0],
            "KPSS_P_Value": kpss_result[1],
            "KPSS_Lags": kpss_result[2],
            "KPSS_Stationary": (
                kpss_result[1] >= significance_level
            )
        })

    return pd.DataFrame(test_results)

In [14]:
# prepare the unique monthly exchange-rate series
monthly_econometric_data = (
    econometric_data[
        [
            "Date",
            "Log_ExchangeRate",
            "ExchangeRate_Log_Change_Pct",
            "ExchangeRate_Positive_Cumulative_Pct",
            "ExchangeRate_Negative_Cumulative_Pct"
        ]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

exchange_rate_test_variables = [
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

exchange_rate_stationarity_results = pd.concat(
    [
        run_stationarity_tests(
            monthly_econometric_data[variable],
            variable
        )
        for variable in exchange_rate_test_variables
    ],
    ignore_index=True
)

exchange_rate_stationarity_results

,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Log_ExchangeRate,Level,105,-1.864525,0.348929,1,False,1.280760,0.010000,6,False
1,Log_ExchangeRate,First difference,104,-8.134200,0.000000,0,True,0.071541,0.100000,1,True
2,ExchangeRate_Log_Change_Pct,Level,105,-8.288868,0.000000,0,True,0.080543,0.100000,1,True
3,ExchangeRate_Log_Change_Pct,First difference,104,-6.269094,0.000000,9,True,0.157573,0.100000,30,True
4,ExchangeRate_Positive_Cumulative_Pct,Level,105,-1.683814,0.439471,1,False,1.578454,0.010000,6,False
5,ExchangeRate_Positive_Cumulative_Pct,First difference,104,-8.252650,0.000000,0,True,0.293281,0.100000,2,True
6,ExchangeRate_Negative_Cumulative_Pct,Level,105,-1.505169,0.530953,0,False,1.586885,0.010000,6,False
7,ExchangeRate_Negative_Cumulative_Pct,First difference,104,-7.863987,0.000000,0,True,0.199321,0.100000,1,True


In [15]:
# test log CPI separately for each food subclass
food_price_stationarity_results = []

for subclass_name, subclass_data in econometric_data.groupby(
    "SubclassDescription"
):
    subclass_results = run_stationarity_tests(
        subclass_data["Log_CPI"],
        "Log_CPI"
    )

    subclass_results.insert(
        0,
        "SubclassDescription",
        subclass_name
    )

    food_price_stationarity_results.append(subclass_results)

food_price_stationarity_results = pd.concat(
    food_price_stationarity_results,
    ignore_index=True
)

food_price_stationarity_results.head(10)

,SubclassDescription,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Baby food,Log_CPI,Level,105,-1.216722,0.666361,7,False,1.486630,0.010000,6,False
1,Baby food,Log_CPI,First difference,104,-1.751753,0.404664,6,False,0.370346,0.089937,2,True
2,Bread and bakery products,Log_CPI,Level,105,-0.203780,0.938070,1,False,1.565412,0.010000,6,False
3,Bread and bakery products,Log_CPI,First difference,104,-6.827378,0.000000,0,True,0.149196,0.100000,4,True
4,Breakfast cereals,Log_CPI,Level,105,-0.678768,0.852173,0,False,1.551419,0.010000,6,False
5,Breakfast cereals,Log_CPI,First difference,104,-10.709926,0.000000,0,True,0.180325,0.100000,1,True
6,Cereals,Log_CPI,Level,105,-1.183943,0.680465,5,False,1.403764,0.010000,6,False
7,Cereals,Log_CPI,First difference,104,-4.254948,0.000531,2,True,0.202674,0.100000,2,True
8,Cheese,Log_CPI,Level,105,0.557774,0.986527,0,False,1.563846,0.010000,6,False
9,Cheese,Log_CPI,First difference,104,-10.890011,0.000000,0,True,0.209836,0.100000,0,True


In [16]:
# summarise stationarity decisions across food subclasses
food_price_stationarity_summary = (
    food_price_stationarity_results
    .groupby("Transformation")
    .agg(
        Food_Subclasses=(
            "SubclassDescription",
            "nunique"
        ),
        ADF_Stationary=(
            "ADF_Stationary",
            "sum"
        ),
        KPSS_Stationary=(
            "KPSS_Stationary",
            "sum"
        )
    )
    .reindex(["Level", "First difference"])
)

food_price_stationarity_summary

,Food_Subclasses,ADF_Stationary,KPSS_Stationary
Transformation,,,
Level,46,0,1
First difference,46,42,44


### Initial Stationarity Interpretation

The log exchange rate is non-stationary in levels but stationary after first differencing according to both the ADF and KPSS tests. It is therefore classified as I(1).

The cumulative depreciation and appreciation components follow the same
pattern. Both are non-stationary in levels and stationary after first
differencing, supporting their inclusion as I(1) variables in the NARDL
framework.

The monthly log exchange-rate change is stationary in levels and is therefore classified as I(0).

For food prices, none of the 46 subclass log CPI series was stationary in levels according to the ADF test, while only one was classified as stationary by the KPSS test. After first differencing, the ADF test classified 42 subclasses as stationary and the KPSS test classified 44 as stationary.

Most food-price series therefore appear to be I(1). However, subclasses for which the tests disagree or fail to establish first-difference stationarity must be examined before model estimation.

In [17]:
# classify food-price integration orders
level_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "Level"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Level_ADF_P_Value",
            "ADF_Stationary": "Level_ADF_Stationary",
            "KPSS_P_Value": "Level_KPSS_P_Value",
            "KPSS_Stationary": "Level_KPSS_Stationary"
        }
    )
)

difference_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "First difference"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Difference_ADF_P_Value",
            "ADF_Stationary": "Difference_ADF_Stationary",
            "KPSS_P_Value": "Difference_KPSS_P_Value",
            "KPSS_Stationary": "Difference_KPSS_Stationary"
        }
    )
)

food_price_integration_status = level_test_results.join(
    difference_test_results
)

level_stationary = (
    food_price_integration_status["Level_ADF_Stationary"]
    & food_price_integration_status["Level_KPSS_Stationary"]
)

difference_stationary = (
    food_price_integration_status["Difference_ADF_Stationary"]
    & food_price_integration_status["Difference_KPSS_Stationary"]
)

food_price_integration_status["Integration_Order"] = np.select(
    [
        level_stationary,
        ~level_stationary & difference_stationary
    ],
    [
        "I(0)",
        "I(1)"
    ],
    default="Requires review"
)

food_price_integration_status[
    "Integration_Order"
].value_counts().to_frame(name="Food_Subclasses")

,Food_Subclasses
Integration_Order,
I(1),40
Requires review,6


### Review of Inconclusive Food-Price Series

A conservative classification requires both tests to support stationarity.

Subclasses are classified as I(0) when both tests indicate stationarity in levels and as I(1) when both tests indicate stationarity after first differencing.

A `Requires review` result does not automatically mean that the series is I(2). It indicates that the two tests disagree or that first-difference stationarity has not yet been established conclusively.

In [18]:
# inspect subclasses with inconclusive results
food_price_series_for_review = (
    food_price_integration_status[
        food_price_integration_status[
            "Integration_Order"
        ] == "Requires review"
    ]
    .sort_values(
        [
            "Difference_ADF_P_Value",
            "Difference_KPSS_P_Value"
        ],
        ascending=False
    )
)

food_price_series_for_review

,Level_ADF_P_Value,Level_ADF_Stationary,Level_KPSS_P_Value,Level_KPSS_Stationary,Difference_ADF_P_Value,Difference_ADF_Stationary,Difference_KPSS_P_Value,Difference_KPSS_Stationary,Integration_Order
SubclassDescription,,,,,,,,,
"Stone fruits and pome fruits, fresh",0.943496,False,0.010000,False,0.525855,False,0.100000,True,Requires review
Baby food,0.666361,False,0.010000,False,0.404664,False,0.089937,True,Requires review
"Salt, condiments and sauces",0.905076,False,0.010000,False,0.348858,False,0.100000,True,Requires review
Other food products n.e.c.,0.858553,False,0.010000,False,0.152017,False,0.100000,True,Requires review
Coffee and coffee substitutes,0.998312,False,0.010000,False,0.001276,True,0.010000,False,Requires review
"Chocolate, cocoa, and cocoa-based food products",0.998883,False,0.010000,False,0.000000,True,0.010000,False,Requires review
